In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    detect_device,
    dropna_splits,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data


In [2]:
DATA_SET = 'rand_B'


df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect


In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
BATCH_SIZE = CFG['BATCH']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  batch=65,536  |  dtype=torch.bfloat16


## Hyperparameters


In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'LR={INIT_LR:.4f} (scaled from {BASE_LR} at batch {BASE_BATCH})  '
      f'BATCH={BATCH_SIZE:,}  WARMUP={WARMUP_EPOCHS} epochs')


MAX_EPOCHS=100  PATIENCE=30  LR=0.0160 (scaled from 0.001 at batch 4096)  BATCH=65,536  WARMUP=5 epochs


## Feature definitions


In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag','vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')


Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Drop rows with NaN and pre-allocate on GPU


In [6]:
required_cols = list(set(ALL_FEATURES + [TARGET]))
df_train, df_val, df_test, nan_stats = dropna_splits(df_train, df_val, df_test, required_cols)
print(f'Rows before: {nan_stats["before"]:,}  after: {nan_stats["after"]:,}  dropped: {nan_stats["dropped"]:,}')

gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Rows before: 1,301,676  after: 1,266,520  dropped: 35,156
Data on GPU  |  VRAM used: 0.53 GB / 85 GB  |  Free: 84.6 GB
Train: 886,455  Val: 253,490  Test: 126,575  Features: 12


## Analytic benchmark


In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 37.0896  RMSE = 0.017118
Coefficients: a = -0.091697, b = -0.156464, c = -0.146662


## Train all feature combinations


In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

print(f'\n{"=" * 70}')
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(f'{"=" * 70}\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 65,536  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=38.5236  Gain=-3.9%  ep=100  10.1s  elapsed=0.2min
  [ 25/130] 3F+iv_lag+rho                  SSE=24.7554  Gain=+33.3%  ep=100  4.4s  elapsed=1.9min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=23.2610  Gain=+37.3%  ep=100  4.5s  elapsed=3.8min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=21.6349  Gain=+41.7%  ep=100  4.4s  elapsed=5.6min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=30.6015  Gain=+17.5%  ep=100  4.4s  elapsed=7.5min
  [125/130] 3F+vix_mom+theta+rho           SSE=35.0944  Gain=+5.4%  ep=100  4.5s  elapsed=9.3min
  [130/130] 3F+theta+vega+rho              SSE=37.3083  Gain=-0.6%  ep=100  4.4s  elapsed=9.7min

Done: 9.7 min for 130 models (avg 4.5s/model)


## Results summary


In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,37.089600,0.000293,0.017118,0.006032,0.001855,0.002182,0.065688,None,None,None
1,3F,38.523643,0.000304,0.017446,0.008688,0.000264,0.005430,0.029563,5.2s,-3.87%,None
2,3F+vix_lag,42.114651,0.000333,0.018241,0.010586,-0.000679,0.007194,-0.060896,4.4s,-13.55%,-9.32%
3,3F+iv_lag,29.957596,0.000237,0.015384,0.009727,0.000535,0.006866,0.245348,4.5s,19.23%,28.87%
4,3F+d_iv_lag,34.440716,0.000272,0.016495,0.010009,0.000181,0.006799,0.132415,4.4s,7.14%,-14.96%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,39.066589,0.000309,0.017568,0.009158,-0.001080,0.005827,0.015886,4.4s,-5.33%,-10.16%
128,3F+gamma+theta+rho,40.955788,0.000324,0.017988,0.009980,0.000243,0.006686,-0.031704,4.5s,-10.42%,-4.84%
129,3F+gamma+vega+rho,41.006966,0.000324,0.017999,0.009988,-0.000023,0.006690,-0.032993,4.5s,-10.56%,-0.12%
130,3F+theta+vega+rho,37.308266,0.000295,0.017168,0.008472,-0.000537,0.005100,0.060180,4.4s,-0.59%,9.02%


## Top 10 by Gain vs Hull-White


In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+d_iv_lag+theta,6,18.5731,0.012113,49.92,4.5
1,3F+iv_lag+d_iv_lag,5,19.8607,0.012526,46.45,4.3
2,3F+iv_lag+d_iv_lag+gamma,6,20.6353,0.012768,44.36,4.4
3,3F+iv_lag+gamma+rho,6,20.7142,0.012793,44.15,4.4
4,3F+iv_lag+gamma+theta,6,20.9088,0.012853,43.63,4.4
5,3F+iv_lag+gamma+vega,6,21.3939,0.013001,42.32,4.4
6,3F+iv_lag+d_iv_lag+vix_mom_lag,6,21.6349,0.013074,41.67,4.4
7,3F+iv_lag+d_iv_lag+rho,6,22.0315,0.013193,40.60,4.3
8,3F+vix_lag+iv_lag+vix_mom,6,22.4737,0.013325,39.41,4.5
9,3F+vix_lag+iv_lag+d_iv_lag,6,22.7083,0.013394,38.77,4.4


## Best per group (3F, +1, +2, +3)


In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f'{nf_label}: {best["combo_name"]}')
    print(f'    SSE={best["SSE"]:.4f}  RMSE={best["RMSE"]:.6f}  Gain={best["Gain_vs_HW_%"]:.2f}%\n')

3F (base): 3F
    SSE=38.5236  RMSE=0.017446  Gain=-3.87%

+1 (4F): 3F+iv_lag
    SSE=29.9576  RMSE=0.015384  Gain=19.23%

+2 (5F): 3F+iv_lag+d_iv_lag
    SSE=19.8607  RMSE=0.012526  Gain=46.45%

+3 (6F): 3F+iv_lag+d_iv_lag+theta
    SSE=18.5731  RMSE=0.012113  Gain=49.92%



## Summary statistics


In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 9.6 min (0.16 hr)
Models trained: 130
Best overall: 3F+iv_lag+d_iv_lag+theta (Gain=49.92%)


## Cleanup


In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.68 GB / 85 GB
Total training time: 9.6 min for 130 models
